# Visualizacao de sequencia Pose3D em video no Jupyter

Este notebook carrega um arquivo `pose3d/pose3d.npz`, renderiza o esqueleto 3D frame a frame e gera um video `.mp4` exibido inline.

A organizacao segue o estilo do `smplx_video_notebook.ipynb`, mas aqui a entrada e `PoseSequence3D` (joints 3D), sem malha SMPL-X.


In [1]:
# Instale dependencias se necessario.
%pip install -q numpy matplotlib imageio imageio-ffmpeg ipython


Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

from pathlib import Path
import json

import imageio.v2 as imageio
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Video, display

np.set_printoptions(precision=4, suppress=True)
plt.rcParams['figure.figsize'] = (8, 8)
plt.rcParams['axes.grid'] = False

DEFAULT_IMUGPT22_JOINT_NAMES = (
    'Pelvis',
    'Left_hip',
    'Right_hip',
    'Spine1',
    'Left_knee',
    'Right_knee',
    'Spine2',
    'Left_ankle',
    'Right_ankle',
    'Spine3',
    'Left_foot',
    'Right_foot',
    'Neck',
    'Left_collar',
    'Right_collar',
    'Head',
    'Left_shoulder',
    'Right_shoulder',
    'Left_elbow',
    'Right_elbow',
    'Left_wrist',
    'Right_wrist',
)

DEFAULT_IMUGPT22_PARENT_INDICES = (
    -1,
    0,
    0,
    0,
    1,
    2,
    3,
    4,
    5,
    6,
    7,
    8,
    9,
    9,
    9,
    12,
    13,
    14,
    16,
    17,
    18,
    19,
)


In [ ]:
# ===== CONFIGURACAO =====
ROOT_DIR = Path.cwd()
# NPZ_PATH = ROOT_DIR / 'pose3d' / 'pose3d.npz'

# Exemplo alternativo do pipeline completo:
NPZ_PATH = ROOT_DIR / 'output' / 'robot_emotions_pose3d' / '10ms' / 'user_02' / 'robot_emotions_10ms_u02_tag11' / 'pose' / 'pose3d' / 'pose3d.npz'

OUT_DIR = ROOT_DIR / 'output' / 'pose3d_notebook_renders'
OUT_VIDEO = OUT_DIR / 'pose3d_sequence.mp4'

FRAME_STEP = 1
MAX_RENDER_FRAMES = 400  # Ex.: 120 para render curto
TRAIL_SECONDS = 1.5
PREVIEW_FRAME = 0

VIEW_ELEV = 18
VIEW_AZIM = -72
CENTER_ON_ROOT = False
FLIP_X = False
SHOW_JOINT_NAMES = False

FIGSIZE = (8.0, 8.0)
DPI = 128
POINT_SIZE = 42
LINE_WIDTH = 3.0


In [4]:
def npz_scalar(value):
    if isinstance(value, np.ndarray) and value.shape == ():
        return value.item()
    return value


def format_path(path: str | Path) -> str:
    path = Path(path)
    try:
        return str(path.relative_to(ROOT_DIR))
    except ValueError:
        return str(path)


def load_pose_sequence(path: str | Path) -> dict[str, object]:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'Arquivo nao encontrado: {path}')

    payload = np.load(path, allow_pickle=True)

    print('Chaves encontradas no arquivo:')
    for key in payload.files:
        value = payload[key]
        print(f'  - {key}: shape={getattr(value, "shape", None)}, dtype={getattr(value, "dtype", None)}')

    if 'joint_positions_xyz' not in payload:
        raise KeyError('O arquivo precisa conter `joint_positions_xyz` com shape (T, J, 3).')

    points = np.asarray(payload['joint_positions_xyz'], dtype=np.float32)
    if points.ndim != 3 or points.shape[2] != 3:
        raise ValueError(f'`joint_positions_xyz` deve ter shape (T, J, 3); recebido {points.shape}.')

    num_frames, num_joints, _ = points.shape

    if 'joint_names_3d' in payload:
        joint_names = [str(v) for v in np.asarray(payload['joint_names_3d']).tolist()]
    elif num_joints == len(DEFAULT_IMUGPT22_JOINT_NAMES):
        joint_names = list(DEFAULT_IMUGPT22_JOINT_NAMES)
    else:
        joint_names = [f'joint_{index:02d}' for index in range(num_joints)]

    if len(joint_names) != num_joints:
        raise ValueError(
            f'Quantidade de joint_names_3d ({len(joint_names)}) difere de joint_positions_xyz ({num_joints}).'
        )

    if 'skeleton_parents' in payload:
        parents = [int(v) for v in np.asarray(payload['skeleton_parents']).tolist()]
    elif num_joints == len(DEFAULT_IMUGPT22_PARENT_INDICES):
        parents = list(DEFAULT_IMUGPT22_PARENT_INDICES)
    else:
        parents = [-1] + [index - 1 for index in range(1, num_joints)]

    if len(parents) != num_joints:
        raise ValueError(f'`skeleton_parents` deve ter {num_joints} entradas; recebido {len(parents)}.')

    fps = float(npz_scalar(payload['fps'])) if 'fps' in payload else None
    if fps is not None and fps <= 0:
        fps = None

    if 'timestamps_sec' in payload:
        timestamps = np.asarray(payload['timestamps_sec'], dtype=np.float32)
        if len(timestamps) != num_frames:
            raise ValueError(
                f'`timestamps_sec` deve ter {num_frames} entradas; recebido {len(timestamps)}.'
            )
    else:
        fps = fps or 20.0
        timestamps = np.arange(num_frames, dtype=np.float32) / float(fps)

    if fps is None:
        if num_frames > 1:
            mean_dt = float(np.mean(np.diff(timestamps)))
            fps = 1.0 / max(mean_dt, 1e-6)
        else:
            fps = 20.0

    clip_id = str(npz_scalar(payload['clip_id'])) if 'clip_id' in payload else path.stem
    coordinate_space = str(npz_scalar(payload['coordinate_space'])) if 'coordinate_space' in payload else 'unknown'

    return {
        'path': path,
        'clip_id': clip_id,
        'joint_names_3d': joint_names,
        'skeleton_parents': parents,
        'joint_positions_xyz': points,
        'timestamps_sec': timestamps,
        'fps': float(fps),
        'num_frames': int(num_frames),
        'num_joints': int(num_joints),
        'coordinate_space': coordinate_space,
    }


def apply_render_transforms(points: np.ndarray, *, flip_x: bool, center_on_root: bool) -> np.ndarray:
    transformed = np.array(points, copy=True)
    if center_on_root:
        transformed -= transformed[:, :1, :]
    if flip_x:
        transformed[..., 0] *= -1.0
    return transformed


def bone_color(joint_name: str) -> str:
    name = joint_name.lower()
    if 'left' in name:
        return '#2f6db3'
    if 'right' in name:
        return '#e58f2a'
    return '#2f855a'


def compute_scene_limits(points: np.ndarray):
    flat = points.reshape(-1, 3)
    mins = np.nanmin(flat, axis=0)
    maxs = np.nanmax(flat, axis=0)
    center = (mins + maxs) / 2.0
    span = float(np.max(maxs - mins))
    span = max(span, 0.8)
    half = span * 0.6
    return tuple((float(c - half), float(c + half)) for c in center)


def set_axes_equal(ax, limits):
    (x_min, x_max), (y_min, y_max), (z_min, z_max) = limits
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_zlim(z_min, z_max)
    if hasattr(ax, 'set_box_aspect'):
        ax.set_box_aspect((x_max - x_min, y_max - y_min, z_max - z_min))


def draw_pose_frame(
    ax,
    frame_points: np.ndarray,
    parents: list[int],
    joint_names: list[str],
    *,
    limits,
    title: str,
    annotate: bool = False,
    trail_points: np.ndarray | None = None,
):
    ax.cla()

    if trail_points is not None and len(trail_points) > 1:
        ax.plot(
            trail_points[:, 0],
            trail_points[:, 1],
            trail_points[:, 2],
            color='#222222',
            linewidth=1.5,
            alpha=0.55,
        )

    for child_index, parent_index in enumerate(parents):
        if parent_index < 0:
            continue
        segment = frame_points[[parent_index, child_index], :]
        ax.plot(
            segment[:, 0],
            segment[:, 1],
            segment[:, 2],
            color=bone_color(joint_names[child_index]),
            linewidth=LINE_WIDTH,
            alpha=0.95,
        )

    ax.scatter(
        frame_points[:, 0],
        frame_points[:, 1],
        frame_points[:, 2],
        c=[bone_color(name) for name in joint_names],
        s=POINT_SIZE,
        depthshade=False,
    )

    if annotate:
        for point, joint_name in zip(frame_points, joint_names):
            ax.text(point[0], point[1], point[2], joint_name, fontsize=7)

    set_axes_equal(ax, limits)
    ax.set_title(title)
    ax.set_xlabel('X (m)')
    ax.set_ylabel('Y (m)')
    ax.set_zlabel('Z (m)')
    ax.view_init(elev=VIEW_ELEV, azim=VIEW_AZIM)
    ax.grid(False)

    for axis in ('xaxis', 'yaxis', 'zaxis'):
        pane = getattr(ax, axis).pane
        pane.set_facecolor((1.0, 1.0, 1.0, 0.0))
        pane.set_edgecolor((0.85, 0.85, 0.85, 1.0))


SEQUENCE = load_pose_sequence(NPZ_PATH)
TRANSFORMED_POINTS = apply_render_transforms(
    SEQUENCE['joint_positions_xyz'],
    flip_x=FLIP_X,
    center_on_root=CENTER_ON_ROOT,
)

render_frame_indices = np.arange(0, len(TRANSFORMED_POINTS), FRAME_STEP, dtype=int)
if MAX_RENDER_FRAMES is not None:
    render_frame_indices = render_frame_indices[:MAX_RENDER_FRAMES]

RENDER_POINTS = TRANSFORMED_POINTS[render_frame_indices]
RENDER_TIMESTAMPS = SEQUENCE['timestamps_sec'][render_frame_indices]
ROOT_POINTS = RENDER_POINTS[:, 0, :]
SCENE_LIMITS = compute_scene_limits(RENDER_POINTS)

sequence_overview = {
    'clip_id': SEQUENCE['clip_id'],
    'coordinate_space': SEQUENCE['coordinate_space'],
    'num_frames_total': SEQUENCE['num_frames'],
    'num_frames_render': int(len(RENDER_POINTS)),
    'num_joints': SEQUENCE['num_joints'],
    'fps': SEQUENCE['fps'],
    'frame_step': FRAME_STEP,
    'center_on_root': CENTER_ON_ROOT,
    'flip_x': FLIP_X,
    'path': format_path(SEQUENCE['path']),
    'out_video': format_path(OUT_VIDEO),
}
print('Resumo da sequencia:')
print(json.dumps(sequence_overview, indent=2, ensure_ascii=False))


Chaves encontradas no arquivo:
  - local_rot_mats: shape=(150, 22, 3, 3), dtype=float32
  - global_rot_mats: shape=(150, 22, 3, 3), dtype=float32
  - posed_joints: shape=(150, 22, 3), dtype=float32
  - root_positions: shape=(150, 3), dtype=float32
  - smooth_root_pos: shape=(150, 3), dtype=float32
  - foot_contacts: shape=(150, 4), dtype=bool
  - global_root_heading: shape=(150, 2), dtype=float32


KeyError: 'O arquivo precisa conter `joint_positions_xyz` com shape (T, J, 3).'

In [ ]:
preview_index = int(np.clip(PREVIEW_FRAME, 0, len(RENDER_POINTS) - 1))
preview_frame_number = int(render_frame_indices[preview_index])

preview_trail = ROOT_POINTS[: preview_index + 1]
if CENTER_ON_ROOT:
    preview_trail = None

fig = plt.figure(figsize=FIGSIZE, dpi=DPI)
ax = fig.add_subplot(111, projection='3d')
draw_pose_frame(
    ax,
    RENDER_POINTS[preview_index],
    SEQUENCE['skeleton_parents'],
    SEQUENCE['joint_names_3d'],
    limits=SCENE_LIMITS,
    title=(
        f"{SEQUENCE['clip_id']} | frame {preview_frame_number + 1}/{SEQUENCE['num_frames']}"
    ),
    annotate=SHOW_JOINT_NAMES,
    trail_points=preview_trail,
)
plt.show()


In [ ]:
def figure_to_rgb_array(fig) -> np.ndarray:
    fig.canvas.draw()
    rgba = np.asarray(fig.canvas.buffer_rgba(), dtype=np.uint8)
    return np.array(rgba[..., :3], copy=True)


def render_pose_video(output_path: str | Path) -> Path:
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    fig = plt.figure(figsize=FIGSIZE, dpi=DPI)
    ax = fig.add_subplot(111, projection='3d')
    render_fps = max(1.0, SEQUENCE['fps'] / max(FRAME_STEP, 1))
    trail_window = max(2, int(round(TRAIL_SECONDS * render_fps)))

    with imageio.get_writer(
        output_path,
        fps=render_fps,
        codec='libx264',
        quality=7,
        ffmpeg_log_level='error',
    ) as writer:
        for local_index, frame_points in enumerate(RENDER_POINTS):
            frame_number = int(render_frame_indices[local_index])

            trail_points = None
            if not CENTER_ON_ROOT:
                trail_start = max(0, local_index - trail_window + 1)
                trail_points = ROOT_POINTS[trail_start : local_index + 1]

            title = (
                f"{SEQUENCE['clip_id']} | "
                f"frame {frame_number + 1}/{SEQUENCE['num_frames']} | "
                f"t={RENDER_TIMESTAMPS[local_index]:.2f}s"
            )
            draw_pose_frame(
                ax,
                frame_points,
                SEQUENCE['skeleton_parents'],
                SEQUENCE['joint_names_3d'],
                limits=SCENE_LIMITS,
                title=title,
                annotate=SHOW_JOINT_NAMES,
                trail_points=trail_points,
            )
            writer.append_data(figure_to_rgb_array(fig))

            if (local_index + 1) % 20 == 0 or (local_index + 1) == len(RENDER_POINTS):
                print(f'Renderizado: {local_index + 1}/{len(RENDER_POINTS)}')

    plt.close(fig)
    return output_path


rendered_path = render_pose_video(OUT_VIDEO)
print(f'Video salvo em: {format_path(rendered_path)}')


In [ ]:
display(Video(str(rendered_path), embed=True))


## Notas de depuracao

- Se o arquivo nao estiver em `pose3d/pose3d.npz`, ajuste `NPZ_PATH` na celula de configuracao.
- Use `FRAME_STEP` e `MAX_RENDER_FRAMES` para depurar mais rapido antes de renderizar tudo.
- `CENTER_ON_ROOT=True` ajuda a analisar gesto sem locomocao global.
- `SHOW_JOINT_NAMES=True` facilita auditoria de mapeamento de joints.
